# Population Work 2: Tax Contribution by Race Over Time

This notebook estimates **tax contribution by race over time** using ACS 5-year data and builds a stacked filled line chart.

Method summary:
- Primary source: ACS aggregate household income by race (`B19025*`).
- Fallback source: if aggregate income is missing, estimate as `median income (B19013*) * households (B11001*)`.
- Tax proxy: estimated effective rate applied to aggregate income.


In [ ]:
import os
import requests
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio

try:
    from census import Census
except Exception:
    Census = None

pd.set_option('display.max_columns', None)
pio.renderers.default = "vscode" if os.environ.get("VSCODE_PID") else "notebook_connected"


In [ ]:
# Config
CENSUS_API_KEY = "942e0a44c121ca03ced84b727df9b004f1f1367d"
YEARS = list(range(2009, 2026))

# Primary + fallback variable mapping
RACE_VARS = {
    'White':    {'agg': 'B19025A_001E', 'median': 'B19013A_001E', 'hh': 'B11001A_001E'},
    'Black':    {'agg': 'B19025B_001E', 'median': 'B19013B_001E', 'hh': 'B11001B_001E'},
    'Asian':    {'agg': 'B19025D_001E', 'median': 'B19013D_001E', 'hh': 'B11001D_001E'},
    'Hispanic': {'agg': 'B19025I_001E', 'median': 'B19013I_001E', 'hh': 'B11001I_001E'},
}

STYLE = {
    'background': '#f5efe2',
    'grid': '#d8cfbf',
    'font': '#2f2a25',
    'colors': {
        'White': '#7aa6c2',
        'Black': '#d98f75',
        'Asian': '#8bb89a',
        'Hispanic': '#e3bf78',
    }
}

census_client = Census(CENSUS_API_KEY) if (Census is not None and CENSUS_API_KEY) else None


In [ ]:
def _to_numeric(df, cols):
    existing = [c for c in cols if c in df.columns]
    if existing:
        df[existing] = df[existing].apply(pd.to_numeric, errors='coerce')
    return df


def fetch_with_census_package(year, fields):
    if census_client is None:
        return None
    try:
        data = census_client.acs5.get(fields, {'for': 'state:*'}, year=year)
        return pd.DataFrame(data)
    except Exception:
        return None


def fetch_with_rest_api(year, fields, api_key):
    base = f"https://api.census.gov/data/{year}/acs/acs5"
    params = {
        'get': ','.join(fields),
        'for': 'state:*',
    }
    if api_key:
        params['key'] = api_key

    try:
        r = requests.get(base, params=params, timeout=30)
        if r.status_code != 200:
            return None
        payload = r.json()
        if not payload or len(payload) < 2:
            return None
        df = pd.DataFrame(payload[1:], columns=payload[0])
        return df
    except Exception:
        return None


def fetch_year_data(year):
    fields = ['NAME']
    for race_map in RACE_VARS.values():
        fields.extend([race_map['agg'], race_map['median'], race_map['hh']])

    # First path: census package
    df = fetch_with_census_package(year, fields)

    # Second path: direct Census REST API
    if df is None or df.empty:
        df = fetch_with_rest_api(year, fields, CENSUS_API_KEY)

    if df is None or df.empty:
        return None

    numeric_cols = [f for f in fields if f != 'NAME']
    df = _to_numeric(df, numeric_cols)

    row = {'Year': year}

    for race, vars_map in RACE_VARS.items():
        agg_var = vars_map['agg']
        median_var = vars_map['median']
        hh_var = vars_map['hh']

        agg_total = df[agg_var].sum(skipna=True) if agg_var in df.columns else np.nan
        if pd.isna(agg_total) or agg_total == 0:
            median_sum = df[median_var].sum(skipna=True) if median_var in df.columns else np.nan
            hh_sum = df[hh_var].sum(skipna=True) if hh_var in df.columns else np.nan
            if pd.notna(median_sum) and pd.notna(hh_sum):
                agg_total = median_sum * hh_sum

        row[f'{race}_AggregateIncome'] = float(agg_total) if pd.notna(agg_total) else np.nan

        # Dynamic effective-rate proxy from median income level (bounded)
        median_mean = df[median_var].mean(skipna=True) if median_var in df.columns else np.nan
        if pd.notna(median_mean):
            eff_rate = np.clip(0.08 + (median_mean / 100000.0) * 0.15, 0.08, 0.28)
        else:
            eff_rate = 0.18

        row[f'{race}_EffRate'] = float(eff_rate)
        row[f'{race}_TaxContribution'] = row[f'{race}_AggregateIncome'] * eff_rate if pd.notna(row[f'{race}_AggregateIncome']) else np.nan

    return row


In [ ]:
# Build time series across available years
records = []
for y in YEARS:
    rec = fetch_year_data(y)
    if rec is not None:
        records.append(rec)

if not records:
    # Local fallback if API is not reachable
    fallback_csv = 'output/tax_contribution_by_race_over_time.csv'
    if os.path.exists(fallback_csv):
        df_tax = pd.read_csv(fallback_csv)
        print(f'Loaded local fallback: {fallback_csv}')
    else:
        raise RuntimeError('No data fetched and no local fallback CSV found.')
else:
    df_tax = pd.DataFrame(records).sort_values('Year').reset_index(drop=True)
    os.makedirs('output', exist_ok=True)
    df_tax.to_csv('output/tax_contribution_by_race_over_time.csv', index=False)
    print('Saved output/tax_contribution_by_race_over_time.csv')

df_tax.head()


Saved output/tax_contribution_by_race_over_time.csv


,Year,White_AggregateIncome,White_EffRate,White_TaxContribution,Black_AggregateIncome,Black_EffRate,Black_TaxContribution,Asian_AggregateIncome,Asian_EffRate,Asian_TaxContribution,Hispanic_AggregateIncome,Hispanic_EffRate,Hispanic_TaxContribution
0,2009,6.560237e+12,0.161444,1.059109e+12,6.225591e+11,0.131813,8.206165e+10,3.683627e+11,0.168404,6.203371e+10,6.889502e+11,0.139823,9.633130e+10
1,2010,6.684937e+12,0.162402,1.085647e+12,6.555950e+11,0.132861,8.710332e+10,4.036935e+11,0.168533,6.803557e+10,7.361838e+11,0.140192,1.032074e+11
2,2011,6.872856e+12,0.164262,1.128948e+12,6.768300e+11,0.133171,9.013402e+10,4.238765e+11,0.171526,7.270590e+10,7.678761e+11,0.141289,1.084924e+11
3,2012,6.945246e+12,0.164918,1.145396e+12,6.858654e+11,0.133509,9.156928e+10,4.399236e+11,0.173315,7.624551e+10,7.870502e+11,0.141284,1.111974e+11
4,2013,6.998733e+12,0.165286,1.156795e+12,6.925425e+11,0.134138,9.289615e+10,4.558065e+11,0.173743,7.919326e+10,8.064370e+11,0.141500,1.141106e+11


In [ ]:
# Tidy for plotting
races = ['White', 'Black', 'Asian', 'Hispanic']
plot_rows = []

for _, r in df_tax.iterrows():
    for race in races:
        val = r.get(f'{race}_TaxContribution', np.nan)
        if pd.notna(val):
            plot_rows.append({'Year': int(r['Year']), 'Race': race, 'TaxContribution': float(val)})

df_plot = pd.DataFrame(plot_rows).sort_values(['Year', 'Race'])
df_plot.head()


,Year,Race,TaxContribution
2,2009,Asian,6.203371e+10
1,2009,Black,8.206165e+10
3,2009,Hispanic,9.633130e+10
0,2009,White,1.059109e+12
6,2010,Asian,6.803557e+10


In [ ]:
# Economist-inspired stacked filled line chart
fig = px.area(
    df_plot,
    x='Year',
    y='TaxContribution',
    color='Race',
    color_discrete_map=STYLE['colors'],
    title='Estimated Tax Contribution by Race Over Time (ACS Years Available)'
)

fig.update_traces(mode='lines', line=dict(width=1.2))
fig.update_layout(
    paper_bgcolor=STYLE['background'],
    plot_bgcolor=STYLE['background'],
    font=dict(color=STYLE['font'], family='Georgia'),
    title_font=dict(size=22),
    xaxis=dict(
        title='Year',
        gridcolor=STYLE['grid'],
        zeroline=False
    ),
    yaxis=dict(
        title='Estimated Tax Contribution (USD)',
        gridcolor=STYLE['grid'],
        tickformat='$,.2s',
        zeroline=False
    ),
    legend_title='Race'
)

os.makedirs('output', exist_ok=True)
fig.write_html('output/tax_contribution_by_race_over_time.html')
print('Generated output/tax_contribution_by_race_over_time.html')
fig.show()


Generated output/tax_contribution_by_race_over_time.html


In [ ]:
# Economist-inspired stacked filled chart: SNAP-based welfare proxy by race over time
WELFARE_VARS = {
    'White':    {'hh_total': 'B22005A_001E', 'hh_assist': 'B22005A_002E', 'median': 'B19013A_001E'},
    'Black':    {'hh_total': 'B22005B_001E', 'hh_assist': 'B22005B_002E', 'median': 'B19013B_001E'},
    'Asian':    {'hh_total': 'B22005D_001E', 'hh_assist': 'B22005D_002E', 'median': 'B19013D_001E'},
    'Hispanic': {'hh_total': 'B22005I_001E', 'hh_assist': 'B22005I_002E', 'median': 'B19013I_001E'},
}


def fetch_welfare_year_data(year):
    fields = ['NAME']
    for race_map in WELFARE_VARS.values():
        fields.extend([race_map['hh_total'], race_map['hh_assist'], race_map['median']])

    df = fetch_with_census_package(year, fields)
    if df is None or df.empty:
        df = fetch_with_rest_api(year, fields, CENSUS_API_KEY)
    if df is None or df.empty:
        return None

    numeric_cols = [f for f in fields if f != 'NAME']
    df = _to_numeric(df, numeric_cols)

    row = {'Year': year}
    for race, vars_map in WELFARE_VARS.items():
        hh_assist = df[vars_map['hh_assist']].sum(skipna=True) if vars_map['hh_assist'] in df.columns else np.nan
        median_income = df[vars_map['median']].mean(skipna=True) if vars_map['median'] in df.columns else np.nan

        # Proxy annual welfare consumption per assisted household with bounded values.
        if pd.notna(median_income):
            annual_benefit = float(np.clip(median_income * 0.18, 2500, 14000))
        else:
            annual_benefit = 6000.0

        row[f'{race}_AssistHH'] = float(hh_assist) if pd.notna(hh_assist) else np.nan
        row[f'{race}_AnnualBenefit'] = annual_benefit
        row[f'{race}_WelfareConsumption'] = row[f'{race}_AssistHH'] * annual_benefit if pd.notna(row[f'{race}_AssistHH']) else np.nan

    return row


welfare_records = []
WELFARE_YEARS = [y for y in YEARS if y <= 2024]
for y in WELFARE_YEARS:
    rec = fetch_welfare_year_data(y)
    if rec is not None:
        welfare_records.append(rec)

if welfare_records:
    df_welfare = pd.DataFrame(welfare_records).sort_values('Year').reset_index(drop=True)
    os.makedirs('output', exist_ok=True)
    df_welfare.to_csv('output/welfare_consumption_by_race_over_time.csv', index=False)
    print('Saved output/welfare_consumption_by_race_over_time.csv')
else:
    fallback_candidates = [
        'output/welfare_consumption_by_race_over_time.csv',
        'welfare_consumption_by_race_over_time.csv',
    ]
    fallback_csv = next((p for p in fallback_candidates if os.path.exists(p)), None)
    if fallback_csv:
        df_welfare = pd.read_csv(fallback_csv)
        print(f'Loaded local fallback: {fallback_csv}')
    else:
        # Keep notebook execution alive when API data is unavailable.
        print('Warning: No welfare data fetched and no local fallback CSV found; skipping welfare chart.')
        df_welfare = pd.DataFrame(columns=['Year'])

welfare_rows = []
races = ['White', 'Black', 'Asian', 'Hispanic']
for _, r in df_welfare.iterrows():
    for race in races:
        val = r.get(f'{race}_WelfareConsumption', np.nan)
        if pd.notna(val):
            welfare_rows.append({'Year': int(r['Year']), 'Race': race, 'WelfareConsumption': float(val)})

df_welfare_plot = pd.DataFrame(welfare_rows)
if not df_welfare_plot.empty:
    df_welfare_plot = df_welfare_plot.sort_values(['Year', 'Race'])

    fig_welfare = px.area(
        df_welfare_plot,
        x='Year',
        y='WelfareConsumption',
        color='Race',
        color_discrete_map=STYLE['colors'],
        title='Estimated SNAP-Linked Welfare Proxy by Race Over Time (ACS Years Available)'
    )

    fig_welfare.update_traces(mode='lines', line=dict(width=1.2))
    fig_welfare.update_layout(
        paper_bgcolor=STYLE['background'],
        plot_bgcolor=STYLE['background'],
        font=dict(color=STYLE['font'], family='Georgia'),
        title_font=dict(size=22),
        xaxis=dict(title='Year', gridcolor=STYLE['grid'], zeroline=False),
        yaxis=dict(title='Estimated SNAP-Linked Welfare Proxy (USD)', gridcolor=STYLE['grid'], tickformat='$,.2s', zeroline=False),
        legend_title='Race'
    )

    fig_welfare.write_html('output/welfare_consumption_by_race_over_time.html')
    print('Generated output/welfare_consumption_by_race_over_time.html')
    fig_welfare.show()
else:
    print('No welfare rows available to plot.')


Saved output/welfare_consumption_by_race_over_time.csv
Generated output/welfare_consumption_by_race_over_time.html


## Notes

- This is an **estimate** of tax contribution, not filed tax returns.
- Primary measure is aggregate household income by race from ACS (`B19025*`).
- Fallback estimate is `median income * households` when aggregate income is unavailable for a year.
- Effective tax rate is modeled from income level and bounded between `8%` and `28%`.
